# Ingestion Workflow - Gestore TeknoService Italia

In [1]:
import io
import re

import pdfplumber
import requests


In [2]:
file_url = "https://www.teknoserviceitalia.com/wp-content/uploads/2025/03/Riciclabolario-generico-Teknoservice-NUOVO.pdf"

response = requests.get(file_url)
response.raise_for_status()

pdf = io.BytesIO(response.content)

# item -> category, in encounter order (dict preserves insertion order in py3.7+)
entries = {}
notes = {}  # item -> extra note text (e.g. "o Ecocentro se in grandi quantità")

DOTLEADER = re.compile(r"^(.*?)\.{6,}\s*(.+)$")
# section-header artifacts: a single letter, optionally followed by its
# decorative lowercase twin, e.g. "B" / "b"
HEADER_ARTIFACT = re.compile(r"^[A-Za-z]$")

## Normalize Waste Categories (to make vocabulary-calendar to match)

In [7]:
def normalize_category(category: str) -> str:
    match category:
        case "Umido":
            return "Organico"
        case "Carta":
            return "Carta e Cartoni"
        case "Plastica":
            return "Imb. Plast. e Metalli"
        case "Vetro":
            return "Vetro"
        case "Indifferenziato":
            return "RUR"
        case _:
            return category

## Text extraction

In [ ]:
loader = pdfplumber.open(pdf)
last_item = None

for page in loader.pages:
    text = page.extract_text() or ""
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        if HEADER_ARTIFACT.match(line):
            # "B" / "b" section markers -> discard, don't let them leak
            # into the next buffer
            continue

        match = DOTLEADER.match(line)
        if match:
            item, category = match.group(1).strip(), match.group(2).strip()
            entries[item] = normalize_category(category)
            last_item = item
        else:
            # continuation / footnote line with no dot-leader, e.g.
            # "o Ecocentro se in grandi quantità" -> attach to the
            # previous item instead of letting it corrupt the next one
            if last_item is not None:
                notes[last_item] = (notes.get(last_item, "") + " " + line).strip()

loader.close()

## Export text to markdown files, split by starting letter (alphabet)

It requires manual changes to the markdown files at the end of the workflow, but still good starting point to automate a little bit

In [9]:
from collections import defaultdict

# Group keys by their first letter
entries_by_letter = defaultdict(list)

for item in entries:
    if item:  # ignore empty keys
        letter = item[0].upper()
        entries_by_letter[letter].append(item)

# Process letters in alphabetical order
for letter in sorted(entries_by_letter):
    # Sort the keys alphabetically within the letter
    items = sorted(entries_by_letter[letter])

    lines = [f"# Vocabolario - Lettera {letter}", "| Oggetto | Conferimento |", "|---|---|"]

    for item in items:
        category = entries[item]
        note = notes.get(item)
        cell = f"{category} ({note})" if note else category
        lines.append(f"| {item} | {cell} |")

    markdown_content = "\n".join(lines)

    # Save this letter's Markdown file before moving to the next letter
    filename = f"../knowledge/sub-ato-e/guide/{letter}.md"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    print(markdown_content)
    print(f"\n... {len(items)} entries starting with {letter}")

# Vocabolario - Lettera A
| Oggetto | Conferimento |
|---|---|
| Abiti usati | Contenitore indumenti usati |
| Accendino | RUR |
| Acetone (contenitore vuoto e lavato) | Imb. Plast. e Metalli |
| Acido | Ecocentro |
| Acquaragia (contenitore) | Ecocentro |
| Addobbo natalizio in carta | Carta e Cartoni |
| Addobbo natalizio sintetico | RUR |
| Adesivo | RUR |
| Ago da cucito | RUR |
| Ago siringa (protetto dal suo cappuccio) | RUR |
| Albero di Natale naturale | Ecocentro |
| Albero di Natale sintetico | Ecocentro |
| Alcool (contenitore) | Imb. Plast. e Metalli |
| Alimenti | Organico |
| Alluminio in fogli, vaschette, lattine (puliti) | Metallo |
| Amianto | Enti autorizzati |
| Ammoniaca (contenitore) | Imb. Plast. e Metalli |
| Antenne | Ecocentro |
| Antiparassitari domestici | Ecocentro |
| Apparecchiatura elettrica | RAEE |
| Armadi | Ecocentro |
| Asciugacapelli | RAEE |
| Aspirapolvere | RAEE |
| Asse da stiro | Ecocentro |
| Assi in legno | Ecocentro |
| Assorbente | RUR |
| 